### Requirements

To run `em_features.ipynb`, make sure the following files are present in the same directory:

- `Feature_generator.xlsx`
- `elements.xlsx`
- CIF files (`*.cif`)

## Generate Compositional Features

In [2]:
import pandas as pd
import numpy as np
from pymatgen.core import Composition


class FormulaVectorizer:
    def __init__(self, element_file="elements.xlsx"):
        self.df = pd.read_excel(element_file, index_col="Symbol")
        self.df.columns = self.df.columns.str.strip()

        self.feature_names = [
            f"{stat}_{prop}"
            for stat in ["avg", "diff", "max", "min", "std"]
            for prop in self.df.columns
        ]

    def featurize(self, formula):
        try:
            comp = Composition(str(formula)).fractional_composition.as_dict()
            elements = [e for e in comp if e in self.df.index]

            if not elements:
                return [np.nan] * len(self.feature_names)

            avg = sum(self.df.loc[e] * comp[e] for e in elements)
            props = self.df.loc[elements]

            return np.concatenate([
                avg.values,
                (props.max() - props.min()).values,
                props.max().values,
                props.min().values,
                props.std(ddof=0).values,
            ])

        except Exception:
            return [np.nan] * len(self.feature_names)


# ------------------------------------------------------------------
# Input files
# ------------------------------------------------------------------

INPUT_FILE = "Feature_generator.xlsx"
ELEMENT_FILE = "elements.xlsx"
OUTPUT_FILE = "To_predict_em.xlsx"

# ------------------------------------------------------------------
# Generate compositional descriptors
# ------------------------------------------------------------------

df = pd.read_excel(INPUT_FILE)
df.columns = df.columns.str.strip()

vectorizer = FormulaVectorizer(ELEMENT_FILE)

feature_df = pd.DataFrame(
    [vectorizer.featurize(f) for f in df["Formula"]],
    columns=vectorizer.feature_names,
)

# ------------------------------------------------------------------
# Exact feature order used during model training
# ------------------------------------------------------------------

columns = [
    "Formula",
    "x",
    "max_metal_ligand_bond_length",
    "SGR No.",
    "mean_metal_ligand_bond_length",
    "vol_per_atom",
    "avg_Martynov-Batsanov EN",
    "polyhedron volume",
    "avg_Thermal conductivity (W/m•K)",
    "avg_Number of outer shell electrons",
    "std_Heat of fusion (kJ/mol)",
    "1/R2",
    "avg_Number of d electrons",
    "std_Number of outer shell electrons",
    "avg_Mulliken EN",
    "avg_First ionization energy (kJ/mol)",
    "c",
    "std_Number of valence electrons",
    "max_Heat atomization (kJ/mol)",
    "std_Atomic weight",
    "distortion_index",
    "std_Thermal conductivity (W/m•K)",
]

# ------------------------------------------------------------------
# Build final table
# ------------------------------------------------------------------

output = pd.DataFrame(index=df.index)
output["Formula"] = df["Formula"]

for col in columns[1:]:
    if col in feature_df:
        output[col] = feature_df[col]
    elif col in df:
        output[col] = df[col]
    else:
        output[col] = np.nan

output = output[columns]
output.to_excel(OUTPUT_FILE, index=False)

print(f"Saved {OUTPUT_FILE}")
print(f"Shape: {output.shape}")

Saved To_predict.xlsx
Shape: (16, 22)


## ### Next Step

Use the generated `To_predict_em.xlsx` file for final prediction after adding the remaining structural features

## Extract Structural Features from CIF Files

In [6]:
import os
import pandas as pd
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer


results = []

for cif in os.listdir():
    if not cif.endswith(".cif"):
        continue

    try:
        structure = Structure.from_file(cif)
        conv = SpacegroupAnalyzer(structure).get_conventional_standard_structure()

        results.append({
            "cif_file": cif,
            "Formula": conv.composition.reduced_formula,
            "SGR No.": SpacegroupAnalyzer(structure).get_space_group_number(),
            "vol_per_atom": conv.volume / conv.composition.num_atoms,
            "c": conv.lattice.c,
        })

    except Exception as e:
        print(f"{cif}: {e}")

df = pd.DataFrame(results)
df.to_excel("CIF_features_for_em_model.xlsx", index=False)

print(f"Saved CIF_features_for_em_model.xlsx ({len(df)} structures)")

Saved CIF_features_for_em_model.xlsx (2 structures)


## Additional Features Required

This script generates 15 of the 20 features required by the emission model. To complete `To_predict_em.xlsx`, the following structural descriptors must be obtained separately using VESTA:

- `max_metal_ligand_bond_length`
- `mean_metal_ligand_bond_length`
- `polyhedron volume`
- `distortion_index`

The `1/R²` descriptor should be calculated using the ionic radii listed in the table provided in the README.